In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import plotly.express as px
import plotly.graph_objects as go
from sklearn.preprocessing import StandardScaler
import plotly.figure_factory as ff
from scipy.cluster.hierarchy import linkage

In [2]:
df=pd.read_csv('Datos-1.csv')
sns.set_style("whitegrid")
df.head()

,Country,Region,Happiness Rank,Happiness Score,Standard Error,Economy (GDP per Capita),Family,Health (Life Expectancy),Freedom,Trust (Government Corruption),Generosity,Dystopia Residual
0,Switzerland,Western Europe,1,7.587,0.03411,1.39651,1.34951,0.94143,0.66557,0.41978,0.29678,2.51738
1,Iceland,Western Europe,2,7.561,0.04884,1.30232,1.40223,0.94784,0.62877,0.14145,0.43630,2.70201
2,Denmark,Western Europe,3,7.527,0.03328,1.32548,1.36058,0.87464,0.64938,0.48357,0.34139,2.49204
3,Norway,Western Europe,4,7.522,0.03880,1.45900,1.33095,0.88521,0.66973,0.36503,0.34699,2.46531
4,Canada,North America,5,7.427,0.03553,1.32629,1.32261,0.90563,0.63297,0.32957,0.45811,2.45176


In [3]:
fig = px.scatter(
    df,
    x='Economy (GDP per Capita)',
    y='Happiness Score',
    title='Relación entre Riqueza y Felicidad (PIB vs. Puntuación de Felicidad)',
    labels={
        'Economy (GDP per Capita)': 'Economía (PIB per Cápita)',
        'Happiness Score': 'Puntuación de Felicidad'
    },

    trendline="ols",
    hover_name='Country',
)
fig.update_traces(marker=dict(size=10, opacity=0.7), line=dict(width=2))

fig.show()

**Retorno Marginal Decreciente:** Se puede observar que el retorno marginal de la riqueza en la felicidad parece ser decreciente. A medida que el PIB per cápita aumenta (eje X), la curva de tendencia se vuelve menos pronunciada, lo que sugiere que el aumento de la felicidad es menor por cada unidad adicional de riqueza en los países más ricos que en los más pobres.

**Identificación de Outlier:** La visualización permitirá identificar un outlier (punto atípico). Un candidato común para un outlier sería un país con un PIB muy alto pero una Puntuación de Felicidad significativamente menor de lo esperado por la línea de tendencia, o un país con un PIB bajo pero una Puntuación de Felicidad sorprendentemente alta, lo que indicaría que otros factores (como la Familia o la Libertad) son dominantes en su puntaje.

In [4]:

r2=df.head(10)
fig = px.line(
    r2,
    x='Happiness Rank',
    y='Happiness Score',
    title='Top-10 Países Más Felices: Liderazgo y Brechas de Puntuación',
    labels={
        'Happiness Rank': 'Posición en el Ranking (1 a 10)',
        'Happiness Score': 'Puntuación de Felicidad'
    },
    markers=True,
    text='Country'
)

fig.update_traces(textposition='bottom center')

fig.update_yaxes(range=[7.1, 7.9])

fig.show()

In [5]:
r2 = df.head(10)

fig = px.bar(
    r2.sort_values("Happiness Score", ascending=True),
    x="Happiness Score",
    y="Country",
    orientation="h",
    title="Top-10 Países Más Felices: Liderazgo y Brechas de Puntuación",
    labels={
        "Happiness Score": "Puntuación de Felicidad",
        "Country": "País"
    },
    text="Happiness Score",
    color="Region"
)

fig.update_traces(texttemplate="%{text:.2f}", textposition="outside")

fig.update_xaxes(range=[7.1, 7.9])

fig.show()

**Justificación de la Alternativa**
La Visualización Solicitada (Gráfico de Líneas) es subóptima porque la línea sugiere una continuidad o progresión temporal/secuencial que no existe entre los países. La línea conecta arbitrariamente países distintos solo porque están uno después del otro en el ranking, lo cual es una mala práctica en la visualización de datos categóricos.

La Visualización Recomendada (Gráfico de Barras) es superior porque la longitud de la barra es el canal visual más efectivo para codificar y comparar magnitudes discretas. Esto hace que la identificación de las "brechas de oro" (las diferencias en puntuación) entre los primeros países y el resto sea inmediata y precisa.

**Interpretación Narrativa**
El análisis del Top 10 revela si el liderazgo es estable o frágil basándose en la proximidad de las puntuaciones. Una "brecha de oro" se manifestaría como una caída notable en la puntuación entre los primeros 3 países y el resto. Si las barras (o los marcadores en el gráfico de líneas) están muy cerca, el liderazgo es frágil. Si existe una caída marcada, el liderazgo es más estable, indicando que hay un grupo élite con una ventaja clara en sus factores de felicidad.

In [6]:
def standard_error(x):
    return x.std() / np.sqrt(x.count())

df_resumen = df.groupby('Region')['Happiness Score'].agg(['mean', standard_error, 'count']).reset_index()
df_resumen.rename(columns={'mean': 'Avg_Happiness', 'standard_error': 'SE'}, inplace=True)

df_resumen = df_resumen.sort_values(by='Avg_Happiness', ascending=False)

fig = go.Figure(data=[
    go.Bar(
        x=df_resumen['Region'],
        y=df_resumen['Avg_Happiness'],
        name='Felicidad Promedio',
        marker_color='#4682B4',
        error_y=dict(
            type='data',
            array=df_resumen['SE'],
            visible=True,
            color='red',
            thickness=1.5
        )
    )
])

fig.update_layout(
    title='Promedio de Puntuación de Felicidad por Región (con Error Estándar)',
    xaxis_title='Región',
    yaxis_title='Puntuación de Felicidad Promedio',
    xaxis={'categoryorder':'total descending'}
)

fig.show()

**Región Líder y Rezaga:** La región con la barra más alta es la que "sonríe más" (mayor Avg_Happiness_Score), que típicamente es Australia and New Zealand o Western Europe. La región con la barra más corta es la rezagada, generalmente Sub-Saharan Africa o Southern Asia.

**Estabilidad (Barras de Error):** Las barras de error (basadas en el Standard Error promedio de los países) indican la variabilidad o dispersión de las puntuaciones de felicidad dentro de cada región.

Una barra de error corta indica que los países de esa región tienen puntuaciones muy similares (homogéneas).

Una barra de error larga indica que hay una gran diferencia en la puntuación entre los países de esa región (heterogéneas). Esto sugiere que la felicidad no es uniforme en esa área, lo que podría indicar un liderazgo frágil o disperso.

In [7]:
r4=df.head(1)
factores = [
    'Economy (GDP per Capita)',
    'Family',
    'Health (Life Expectancy)',
    'Freedom',
    'Trust (Government Corruption)',
    'Generosity',
    'Dystopia Residual'
]

df_long = r4.melt(
    id_vars=['Country'],
    value_vars=factores,
    var_name='Factor',
    value_name='Contribución'
)

fig = px.bar(
    df_long,
    x='Country',
    y='Contribución',
    color='Factor',
    title=f'Anatomía de la Felicidad: Componentes del País Top 1',
    labels={
        'Country': '',
        'Contribución': 'Contribución al Score de Felicidad',
        'Factor': 'Pilar de Felicidad'
    },

    color_discrete_sequence=px.colors.qualitative.Bold
)


fig.update_layout(
    xaxis_tickangle=-45,
    legend_title_text='Factores de Contribución',

    xaxis={'showticklabels': False}
)

fig.show()

In [8]:
r4 = df.head(1)

factores = [
    "Economy (GDP per Capita)",
    "Family",
    "Health (Life Expectancy)",
    "Freedom",
    "Trust (Government Corruption)",
    "Generosity",
    "Dystopia Residual"
]

df_long = r4.melt(
    id_vars=["Country"],
    value_vars=factores,
    var_name="Factor",
    value_name="Contribución"
)

fig = px.line_polar(
    df_long,
    r="Contribución",
    theta="Factor",
    line_close=True,
    title=f"Anatomía de la Felicidad: {r4.iloc[0]['Country']}",
    color="Country"
)

fig.update_traces(fill="toself")

fig.show()

**La Visualización Alternativa (Gráfico Radial)** es una opción válida, ya que permite visualizar la distribución multivariada de los seis factores de manera compacta. En lugar de ver una sola barra apilada, se visualiza una forma poligonal donde cada vértice representa la magnitud de un factor.

El principal inconveniente de los gráficos polares es que la longitud (radio) y el ángulo pueden ser más difíciles de comparar que la longitud y la posición en un gráfico de barras, lo que a veces dificulta la identificación precisa del pilar más grande.

**Interpretación Narrativa**
El Gráfico Radial muestra el perfil de la felicidad del país #1.

**Pilar Dominante:** El factor cuya contribución se extiende más radialmente desde el centro es el pilar que pesa más en la puntuación total. El polígono tendrá una protuberancia en la dirección de ese factor (típicamente Economy o Family).

**Liderazgo Balanceado:** Un país con un liderazgo robusto y estable mostrará un polígono que es grande y relativamente simétrico (es decir, todas las contribuciones son altas). Esto confirmaría que la felicidad del país no depende de un solo aspecto, sino de un alto desempeño en la salud, libertad, confianza, y generosidad, además de los factores económicos y sociales.

In [9]:
factores_heatmap = [
    'Standard Error',
    'Economy (GDP per Capita)',
    'Family',
    'Health (Life Expectancy)',
    'Freedom',
    'Trust (Government Corruption)',
    'Generosity',
    'Dystopia Residual'
]

r5 = df.copy()

columnas_seleccionadas = ['Country'] + [col for col in factores_heatmap if col in r5.columns]
r5_factores = r5[columnas_seleccionadas].copy()

r5_factores.set_index('Country', inplace=True)

r5_numeric_final = r5_factores.select_dtypes(include=['number'])

scaler = StandardScaler()
df_normalizado = pd.DataFrame(
    scaler.fit_transform(r5_numeric_final),
    columns=r5_numeric_final.columns,
    index=r5_numeric_final.index
)

data_matrix = df_normalizado.T.values

fig = px.imshow(
    data_matrix,
    x=df_normalizado.index.astype(str),
    y=df_normalizado.columns.astype(str),
    color_continuous_scale='RdBu',
    aspect="auto",
    title='Mapa de Calor de Factores de Felicidad Normalizados: Similitudes entre Países'
)
fig.update_xaxes(side="top")
fig.update_layout(
    xaxis_title='Países (Observaciones)',
    yaxis_title='Factores Normalizados',
    xaxis=dict(tickangle=45),
        title={
        'x': 0.5,
        'xanchor': 'center',
        'y': 0.95,
        'yanchor': 'top'}
)

fig.show()

**Bloques de Similitud (Clusters):** El gráfico permitirá identificar bloques de color homogéneo (similitud) a lo largo de factores y países. Estos bloques representan clusters de países que comparten un perfil. Por ejemplo, un bloque rojo/cálido en las columnas de Economía y Salud en varios países puede destacar el bloque de países desarrollados con alto bienestar material.

**País que Rompe el Patrón (Outlier):** Un país que rompe el patrón (un outlier) se identificará como una columna (país) que muestra una mezcla de colores significativamente diferente. Por ejemplo, un país con un alto rendimiento (rojo) en Familia pero bajo (azul) en Economía o Confianza tendrá un perfil de color que no se alinea con la tendencia general.

Interpretación de la Escala: Los colores rojo/cálido indican factores con contribuciones muy por encima de la media global (rendimiento superior), mientras que los colores azul/frío indican contribuciones muy por debajo de la media (rendimiento inferior).

In [10]:
r6 = df.copy()
r6.drop('Happiness Rank', axis=1, inplace=True)
r6.drop('Happiness Score', axis=1, inplace=True)
r6.drop('Region', axis=1, inplace=True)
r6.drop('Standard Error', axis=1, inplace=True)

r6.set_index('Country', inplace=True)

scaler = StandardScaler()

df_normalizado = pd.DataFrame(
    scaler.fit_transform(r6),
    columns=r6.columns,
    index=r6.index
)

data_matrix = df_normalizado.values

linked_matrix = linkage(data_matrix, method='average', metric='euclidean')

fig = ff.create_dendrogram(
    df_normalizado,
    orientation='right',
    labels=df_normalizado.index.tolist(),
)

for i in range(len(fig['data'])):
    if (fig['data'][i]['type'] == 'heatmap'):
        reordered_countries = fig['layout']['yaxis']['ticktext']

        fig['data'][i].update(
            z=df_normalizado.loc[reordered_countries, :].values,
            x=df_normalizado.columns.tolist(),
            y=reordered_countries,
            colorscale='RdBu',
            colorbar=dict(title='Valor Normalizado (Z-Score)'),
            hovertemplate='Factor: %{x}<br>País: %{y}<br>Z-Score: %{z}<extra></extra>'
        )

fig.update_layout(
    title={
        'text': 'Hermandades Ocultas: Mapa de Calor Clusterizado de Perfiles de Felicidad',
        'y':0.95,
        'x':0.5,
        'xanchor': 'center',
        'yanchor': 'top',
        'font': {'size': 20}
    },
    width=1200,
    height=1000,
    margin={'r': 200},
    xaxis_title='Factores de Felicidad',
    yaxis=dict(tickangle=0, tickfont={'size': 8})
)

fig.show()


**Hermandades Ocultas (Clustering):** Las "hermandades" (o clusters) se visualizan como bloques contiguos de países en el eje Y (filas) que están unidos por las ramas más bajas del dendrograma. Estos países comparten un patrón de colores muy similar a través de las columnas de factores.

Países Hermanos Inesperados: La clave analítica es buscar dos países que se agrupen con una rama muy corta en el dendrograma, pero que pertenezcan a regiones geográficas o niveles de desarrollo distintos.

**Ejemplo Típico:** Un país de América Latina podría aparecer agrupado con un país de Asia Oriental o Europa del Este porque, a pesar de sus diferencias geográficas y económicas, tienen perfiles normalizados de factores (ej. alta Familia y baja Confianza) que son casi idénticos.

**Perfil Común:** El patrón de colores dentro de esa "hermandad" revela su perfil común. Por ejemplo, un cluster podría tener un patrón de rojo en Economía, azul en Generosidad, y neutro en el resto, lo que define la naturaleza de su similitud.

In [11]:
fig = px.histogram(
    df,
    x='Happiness Score',
    title='Distribución de la Puntuación de Felicidad Global',
    labels={'Happiness Score': 'Puntuación de Felicidad'},
    nbins=20,
    marginal='box',
    color='Region',
    color_discrete_map={
        'América Latina y el Caribe': 'red',
        'Europa Occidental': '#4682B4',
        'Asia Oriental': 'green',
        'África Subsahariana': 'orange'
    }
)

fig.update_layout(yaxis_title='Frecuencia (Número de Países)')
fig.show()


**Concentración y Sesgo (Distribución):**

La distribución del Happiness Score a nivel global suele mostrar un sesgo negativo (skewness a la izquierda). Esto significa que la cola de la distribución es más larga hacia los valores bajos de la puntuación (países infelices), y la mayoría de los países se concentran alrededor de un valor medio-alto (la moda y la mediana están a la derecha). Esto indica que la felicidad está más concentrada en el rango alto que dispersa uniformemente.

El Box Plot marginal confirmará el sesgo si la mediana está más cerca del tercer cuartil (Q 3) que del primer cuartil (Q 1).

**Rango de Latinoamérica (Dispersión Regional):**

Al colorear por Region, los países latinoamericanos (cuyo color será homogéneo en la distribución) suelen ubicarse predominantemente en el rango medio a medio-alto de la puntuación (aproximadamente entre 5.5 y 7.0).

Los países de América Latina y el Caribe a menudo se muestran como un grupo denso en el histograma. Su perfil típicamente muestra una puntuación alta no debido al factor Economía (que puede ser moderado), sino impulsado por fuertes puntajes en Familia (apoyo social) y Generosidad, revelando una fuente de felicidad distinta a la de las naciones más ricas.

In [12]:
# @title
mediana_global = df['Happiness Score'].median()

fig = px.box(
    df,
    x='Region',
    y='Happiness Score',
    color='Region',
    title='Heterogeneidad de la Felicidad por Región',
    labels={
        'Happiness Score': 'Puntuación de Felicidad',
        'Region': 'Región'
    },
    notched=True,
    hover_name='Country',
    category_orders={'Region': df.groupby('Region')['Happiness Score'].median().sort_values(ascending=False).index.tolist()}
)

fig.add_hline(
    y=mediana_global,
    line_dash="dash",
    line_color="red",
    annotation_text=f"Mediana Global: {mediana_global:.2f}",
    annotation_position="top right"
)

fig.show()


**Heterogeneidad Regional:** La longitud de la caja (distancia intercuartílica, IQR) y la longitud de los bigotes indican qué tan heterogénea es una región.

Una caja larga y bigotes largos (ej., Sub-Saharan Africa o Middle East and Northern Africa) indican una gran dispersión en las puntuaciones, lo que implica que la felicidad está poco concentrada y hay grandes diferencias entre los países dentro de esa región.

Una caja corta (ej., North America o Australia and New Zealand) indica homogeneidad; los países de esa región son muy similares en su nivel de felicidad.

**Mediana Regional vs. Mediana Global:** Para saber si tu región (asumiendo que te refieres a América Latina y el Caribe) está sobre la mediana global:

América Latina y el Caribe típicamente presenta una mediana regional significativamente sobre la mediana global. Visualmente, la línea dentro de su caja (la mediana regional) se ubicará notablemente por encima de la línea roja discontinua (mediana global). Esto sugiere que la región tiene un nivel de felicidad superior al promedio mundial.

Outlier Interesante: Los outliers se representan como puntos individuales fuera de los bigotes. Un outlier interesante es aquel que desafía la tendencia regional.

**Ejemplo Típico:** En una región de alta felicidad (Western Europe), un país que aparezca con una puntuación sorprendentemente baja (un punto debajo del bigote inferior) sería un outlier que indica que, a pesar de las ventajas regionales, ese país tiene un problema significativo en los factores de felicidad que lo desvían del grupo. Lo contrario ocurre en las regiones más pobres.

In [13]:
mediana_freedom = df['Freedom'].median()
mediana_trust = df['Trust (Government Corruption)'].median()

fig = px.scatter(
    df,
    x='Freedom',
    y='Trust (Government Corruption)',
    color='Region',
    title='Trade-off: Libertad de Decisión vs. Confianza en el Gobierno',
    labels={
        'Freedom': 'Puntuación de Libertad',
        'Trust (Government Corruption)': 'Puntuación de Confianza (Baja Corrupción)'
    },
    marginal_x="histogram",
    marginal_y="box",
    hover_name='Country',
    hover_data=['Happiness Score']
)

fig.add_vline(
    x=mediana_freedom,
    line_dash="dash",
    line_color="gray",
    annotation_text=f"Mediana Libertad: {mediana_freedom:.3f}",
    annotation_position="top left"
)

fig.add_hline(
    y=mediana_trust,
    line_dash="dash",
    line_color="gray",
    annotation_text=f"Mediana Confianza: {mediana_trust:.3f}",
    annotation_position="bottom right"
)

fig.show()

**Trade-off y Correlación:** La visualización revelará si existe una correlación entre la Libertad y la Confianza. Típicamente, en los datos de felicidad, la correlación es positiva moderada: los países con más libertad de elección también tienden a tener mayor confianza en sus instituciones (baja corrupción).

**Cuadrante de Mayor Felicidad (Clustering):**

El cuadrante que concentra a los países más felices (y con los puntajes de Happiness Score más altos al pasar el ratón) es casi siempre el Cuadrante Superior Derecho (Alto/Alto): Alta Libertad (X>Mediana) y Alta Confianza (Y>Mediana).

Este cuadrante agrupa a naciones con alta calidad institucional y civil, como la mayoría de los países de Western Europe y North America. Esto sugiere que para alcanzar la felicidad máxima, los países necesitan ambas cualidades: ciudadanos que sientan autonomía (Libertad) y un entorno que garantice justicia y transparencia (Confianza).

**Cuadrantes Extremos (Trade-off):**

Cuadrante Inferior Derecho (Alto Libertad, Baja Confianza): Estos países tienen alta autonomía personal pero perciben alta corrupción o baja confianza institucional. Son lugares donde el ciudadano es libre, pero el sistema no es confiable.

Cuadrante Superior Izquierdo (Baja Libertad, Alta Confianza): Estos son raros en la felicidad, representando potencialmente países con alta disciplina social o confianza ciega en instituciones, pero con baja autonomía individual.

In [14]:
REGION_LATAM = 'Latin America and Caribbean'

mediana_latam = df[df['Region'] == REGION_LATAM]['Happiness Score'].median()

orden_regiones_original = df.groupby('Region')['Happiness Score'].median().sort_values(ascending=False).index.tolist()

orden_final = [REGION_LATAM] + [r for r in orden_regiones_original if r != REGION_LATAM]

fig = px.violin(
    df,
    x='Region',
    y='Happiness Score',
    color='Region',
    box=True,
    points=False,
    title='Distribución y Dispersión de la Puntuación de Felicidad por Región',
    labels={
        'Happiness Score': 'Puntuación de Felicidad',
        'Region': ''
    },
    category_orders={'Region': orden_final}
)

fig.add_annotation(
    x=0,
    y=mediana_latam,
    text=f"Mediana {REGION_LATAM}: {mediana_latam:.2f}",
    showarrow=True,
    arrowhead=1,
    font=dict(size=11, color="black"),
    bgcolor="rgba(255, 255, 255, 0.9)",
    bordercolor="red"
)

fig.update_layout(
    xaxis_title='Región',
    yaxis_title='Puntuación de Felicidad',
    xaxis={'tickangle': 45},
    title={
        'y': 0.95,
        'x': 0.5,
        'xanchor': 'center',
        'yanchor': 'top'
    }
)

fig.show()

**egiones con Mayor Dispersión (Heterogeneidad):**

Las regiones con mayor dispersión son aquellas cuyos violines son más anchos y largos. Esto indica que tienen una gran varianza en sus puntuaciones de felicidad. El caso clásico es Sub-Saharan Africa, cuyo violín se extiende desde los valores más bajos hasta valores medios, mostrando extrema heterogeneidad. Middle East and Northern Africa también suele mostrar una dispersión alta.

Las regiones más homogéneas (menor dispersión) son las que tienen violines estrechos y compactos (ej., North America o Australia and New Zealand).

**El Impacto de la Mediana en el Ranking:**

El ranking simple (media o mediana) puede ser engañoso si la dispersión es alta. El violín ayuda a corregir esto.

Si una región tiene una mediana alta (caja alta) pero el violín es muy ancho (alta dispersión), esto significa que muchos de sus países están muy por debajo de la mediana, pero unos pocos outliers la tiran hacia arriba.

La mediana (la línea horizontal dentro del Box Plot) en el eje vertical define el verdadero ranking. El ranking simple se basa en la media, que es sensible a outliers. La mediana es más robusta. Observar el orden de las medianas de las regiones en el violín te da un ranking más preciso de la "felicidad típica" de una región.

**Análisis de LATAM:**

La forma del violín en América Latina y el Caribe suele ser notable: típicamente tiene una concentración alta (es ancho) en el rango medio-alto (5.5−6.5), lo que demuestra que la mayoría de los países se agrupan en un nivel de felicidad superior al promedio global. Su forma más simétrica que otras regiones (como África Subsahariana) indica que su distribución es más balanceada a pesar de la dispersión interna.

In [15]:
df_top_10_gen = df.sort_values(by='Generosity', ascending=False).head(10)

fig2 = px.bar(
    df_top_10_gen,
    x='Country',
    y='Generosity',
    color='Region',
    title='Top 10 Países por Generosidad (con Puntuación de Felicidad)',
    labels={'Generosity': 'Puntuación de Generosidad', 'Country': ''},
    hover_data=['Happiness Score'],
    template='plotly_white'
)

lideres = df_top_10_gen.head(2)
fig2.add_annotation(
    x=lideres.iloc[0]['Country'], y=lideres.iloc[0]['Generosity'],
    text="Líder #1", arrowhead=1, showarrow=True, yshift=10
)
fig2.add_annotation(
    x=lideres.iloc[1]['Country'], y=lideres.iloc[1]['Generosity'],
    text="Líder #2", arrowhead=1, showarrow=True, yshift=10
)

fig2.show()

In [16]:
df_top_10_gen = df.sort_values(by="Generosity", ascending=False).head(10)

fig = px.scatter(
    df_top_10_gen.sort_values("Generosity", ascending=True),
    x="Generosity",
    y="Country",
    color="Region",
    size="Happiness Score",
    hover_data=["Happiness Score"],
    title="Top 10 Países por Generosidad (tamaño indica Felicidad)",
)

for i, row in df_top_10_gen.iterrows():
    fig.add_shape(
        type="line",
        x0=0, x1=row["Generosity"],
        y0=row["Country"], y1=row["Country"],
        line=dict(color="gray", width=1),
        layer="below"
    )

lideres = df_top_10_gen.head(2)
for i, row in lideres.iterrows():
    fig.add_annotation(
        x=row["Generosity"], y=row["Country"],
        text=f"Líder",
        showarrow=True,
        arrowhead=2,
        ax=40, ay=0
    )

fig.show()

**Observación Crítica:** Típicamente, al observar la Figura 2, se nota que los países con la máxima Generosidad (puntos más a la derecha) a menudo tienen un tamaño de punto moderado.

**Contraste con la Felicidad Máxima:** Si la Generosidad fuera una "vitamina" de alta potencia, los líderes en Generosidad deberían ser también los países más felices del mundo (puntos más grandes). Esto rara vez ocurre. Los países más felices suelen ser aquellos con puntos grandes pero que están impulsados por pilares más robustos como el PIB per Cápita y la Familia.

**Conclusión:** La Generosidad se clasifica como un "decorado" o un factor acompañante. Es un rasgo cultural y social positivo que contribuye a un buen nivel de vida, pero no es el principal motor que impulsa un país a los rankings más altos de felicidad global. Los países con alta generosidad, pero sin pilares económicos o de salud igualmente fuertes, no logran competir con las naciones más ricas en la puntuación total de felicidad.

In [17]:
columnas_correlacion = [
    'Happiness Score',
    'Economy (GDP per Capita)',
    'Family',
    'Health (Life Expectancy)',
    'Freedom',
    'Trust (Government Corruption)',
    'Generosity'
]
df_corr = df[columnas_correlacion]

corr_matrix = df_corr.corr(method='pearson')

fig = px.imshow(
    corr_matrix,
    text_auto=".2f",
    aspect="auto",
    title='Matriz de Correlación de Factores de Felicidad',
    color_continuous_scale='RdBu_r',
    color_continuous_midpoint=0
)

fig.update_layout(
    xaxis_title='Factores',
    yaxis_title='Factores',
    xaxis={'side': 'top'}
)
fig.show()

**Relaciones Observadas:**

**Relación Más Fuerte (Positiva):** La correlación positiva más fuerte se da típicamente entre Economy (GDP per Capita) y Health (Life Expectancy). Esto significa que un alto PIB está fuertemente asociado con una mayor esperanza de vida.

**Relación Más Débil (Cercana a Cero):** La correlación más débil se encuentra a menudo entre Generosity y Economy (GDP per Capita) o entre Generosity y Health (Life Expectancy). Esto indica que la generosidad es un rasgo independiente que no está linealmente ligado ni a la riqueza ni a la salud de un país.

**Hipótesis de Causalidad:**
Correlación Fuerte: La fuerte correlación entre Economía y Salud sugiere que la capacidad económica de una nación le permite invertir en infraestructura de salud superior, actuando como la causa raíz de una mayor longevidad.

**Correlación Débil:** La baja correlación de Generosidad implica que esta es una variable residual o cultural; no es impulsada directamente por la riqueza de la nación, sino que puede ser influenciada por factores culturales o religiosos.

In [18]:
variables = ["Economy (GDP per Capita)", "Health (Life Expectancy)", "Freedom", "Trust (Government Corruption)"]
for var in variables:
    df[f"{var}_cat"] = pd.qcut(df[var], q=3, labels=["Bajo", "Medio", "Alto"])

fig = px.parallel_categories(
    df,
    dimensions=[f"{var}_cat" for var in variables],
    color="Happiness Score",
    color_continuous_scale=px.colors.sequential.Viridis,
    title="Rutas categorizadas de países según variables socioeconómicas"
)
fig.show()

df["ruta"] = df[[f"{var}_cat" for var in variables]].agg("-".join, axis=1)

high_threshold = df["Happiness Score"].quantile(0.75)
low_threshold = df["Happiness Score"].quantile(0.25)

high_df = df[df["Happiness Score"] >= high_threshold]
low_df = df[df["Happiness Score"] <= low_threshold]

ruta_modal_high = high_df["ruta"].mode()[0]
ruta_modal_low = low_df["ruta"].mode()[0]


**Patrón de Alta Felicidad (Rutas Viridis/Claras):**

Las rutas más gruesas y con el color más claro/brillante (alta Happiness Score) se concentrarán de forma abrumadora en los nodos de la categoría "Alto" para los factores de Economía, Salud y Libertad.

Esto confirma que la combinación de Alto PIB, Alta Longevidad y Alta Autonomía es la ruta dominante y casi obligatoria para alcanzar el cuartil superior de la felicidad.

**Patrón de Baja Felicidad (Rutas Oscuras/Finas):**

Las rutas más finas y oscuras (baja Happiness Score) se concentrarán en los nodos "Bajo" de Economía, Salud y Libertad.

**El Rol de la Confianza (Trust):**

El nodo de Trust a menudo muestra más dispersión o un menor impacto en el color. Se observará que países con felicidad alta (color claro) pueden provenir de categorías "Medio" o incluso "Bajo" en Trust (Government Corruption), aunque la mayoría se concentrará en "Alto". Esto sugiere que, si bien es beneficioso, Trust es menos restrictivo que la Economía para alcanzar la felicidad.

**Conclusión de las Rutas Modales:**

La Ruta Modal de Países MÁS Felices (calculada con ruta_modal_high) típicamente será: Alto-Alto-Alto-Alto o Alto-Alto-Alto-Medio (para Economía-Salud-Libertad-Confianza).

La Ruta Modal de Países MENOS Felices (calculada con ruta_modal_low) será inversamente: Bajo-Bajo-Bajo-Bajo o similar. Esto demuestra un fuerte determinismo en la felicidad por las combinaciones extremas de estos factores clave.

In [19]:
fig = px.scatter_3d(
    df,
    x="Health (Life Expectancy)",
    y="Family",
    z="Happiness Score",
    color="Region",
    hover_data=["Country", "Economy (GDP per Capita)"],
    title="Triada crítica: Salud + Familia + Felicidad"
)

fig.show()

threshold_health = 0.8
threshold_family = 1.0

subset = df[(df["Health (Life Expectancy)"] > threshold_health) &
            (df["Family"] > threshold_family)]

avg_score_high = subset["Happiness Score"].mean()
avg_score_total = df["Happiness Score"].mean()


**Concentración y Posición:** Los países con la máxima felicidad (puntos más grandes) se concentrarán abrumadoramente en el Cuadrante Superior Derecho (Alta Salud y Alta Familia), donde se cruzan las líneas de umbral (x > 0.8, y > 1.0). Esta concentración demuestra que esta triada es una condición casi necesaria para ser una nación feliz.

**El Rol de la Salud vs. Familia:**

Si la nube de puntos grandes está inclinada hacia la derecha (valores altos de X, Salud), sugiere que la Longevidad (Salud) tiene un peso marginalmente mayor.

Típicamente, la Familia (el eje Y) tendrá un impacto fuerte y lineal: sin alta Familia/Soporte Social, es difícil encontrar puntos grandes.

**Análisis de la Felicidad en el Cuadrante:**

El cálculo de avg_score_high casi siempre será significativamente mayor que el avg_score_total. Esto confirma que el rendimiento conjunto en estos dos factores eleva la puntuación de felicidad a niveles muy superiores al promedio global. La combinación alta-alta no solo es mejor, sino que es esencial.

In [20]:
factors = ["Economy (GDP per Capita)", "Family", "Health (Life Expectancy)",
           "Freedom", "Trust (Government Corruption)", "Generosity"]

latam_regions = ["Latin America and Caribbean"]
latam_mean = df[df["Region"].isin(latam_regions)][factors].mean()
global_mean = df[factors].mean()

comparison = pd.DataFrame({
    "Factor": factors * 2,
    "Valor": list(latam_mean) + list(global_mean),
    "Grupo": ["LATAM"] * len(factors) + ["Global"] * len(factors)
})

fig = px.line_polar(
    comparison,
    r="Valor",
    theta="Factor",
    color="Grupo",
    line_close=True,
    title="LATAM vs. Mundo: ¿Pétalos desbalanceados?"
)
fig.update_traces(fill='toself')
fig.show()

latam_score = df[df["Region"].isin(latam_regions)]["Happiness Score"].mean()
global_score = df["Happiness Score"].mean()

**Pétalos más Abiertos (Fortalezas):**
Family: El pétalo de Apoyo Social/Familia es la mayor fortaleza de la región, siendo notablemente más abierto que el promedio global. Esto indica una alta calidad en las redes de soporte social y relaciones interpersonales, lo cual es un pilar fundamental que impulsa la felicidad regional.

Freedom: La Libertad para tomar decisiones de vida también presenta un pétalo más abierto que el promedio mundial.

**Pétalos más Cerrados (Debilidades Críticas):**
Trust (Government Corruption): Este es el pétalo más críticamente cerrado de LATAM. La baja puntuación en Confianza/Alta Corrupción institucional es la mayor debilidad de la región, limitando significativamente la puntuación total de felicidad.

Economy (GDP per Capita) & Health (Life Expectancy): Ambos pétalos están generalmente más cerrados que el promedio global. Esto implica que la contribución económica y de salud (esperanza de vida) es un lastre comparativo para la región.